In [2]:
import nibabel as nib
import torch
import matplotlib.pyplot as plt
from generator import DiffeoGenerator
from datasets import ADNIDataset, MultiEpochsDataLoader
import os
import numpy as np
import random

import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

2025-03-02 01:57:55.236618: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-02 01:57:55.266768: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
dataset = ADNIDataset("/scratch/anirudhkaushik/ADNI1_Complete_Updated.csv", 'train', aug_p=0.8)
dataloader = MultiEpochsDataLoader(dataset,
                                    batch_size=4,
                                    drop_last=False,
                                    shuffle=True,
                                    num_workers=4,
                                    pin_memory=True,
                                    prefetch_factor=1)
print(len(dataloader))

546


In [ ]:
# load model  from /scratch/anirudhkaushik/CounterSynth_models/generator_38.pth
nb_features = [
        [16, 32, 32, 32, 32],  # encoder
        [32, 32, 32, 32, 32, 32, 16, 16]  # decoder
    ]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DiffeoGenerator(inshape=(128, 128, 128),
                              c_dim=2,
                              use_probs=False,
                              int_downsize=2,
                              in_channels=1,
                              int_steps=7,
                              nb_unet_features=nb_features).to(device)

model = torch.nn.DataParallel(model)
model.load_state_dict(torch.load('/scratch/anirudhkaushik/saved_models/generator_1080.pth'))
model.eval()



In [6]:
with tqdm(total=len(dataloader.dataset)) as progress_bar:
        for i, (imgs, class_labels, reg_labels) in enumerate(dataloader):
            # Convert to float32
            x_real = Variable(imgs.type(torch.float32)).to(device)
            class_labels = Variable(class_labels.type(torch.float32)).to(device)
            reg_labels = Variable(reg_labels.type(torch.float32)).to(device)

            # Sample transformation by shuffling labels
            rand_idx = torch.randperm(class_labels.size(0))
            sampled_class_labels = class_labels[rand_idx]
            sampled_reg_labels = reg_labels[rand_idx]
            diff_labels = (sampled_reg_labels - reg_labels)
            target_labels = torch.cat([sampled_class_labels, sampled_reg_labels], dim=1)

            x_real.requires_grad_()
            x_fake, preint_flow = model(x_real, target_labels)

            break

  0%|          | 0/2181 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 58102, 58166, 58167, 58227) exited unexpectedly

In [ ]:
plt.figure(figsize=(15, 15))


img_org = x_real[0, 0].detach().cpu().numpy()
plt.subplot(1,2,1)
plt.title(f"{reg_labels[0].item()}: Original")
plt.imshow(img_org[:, :, 64], cmap='gray')
plt.subplot(1,2,2)
plt.title(f"{sampled_reg_labels[0].item()}: Generated")
plt.imshow(x_fake[0, 0, :, :, 64].detach().cpu().numpy(), cmap='gray')
plt.show()
print(reg_labels[rand_idx], reg_labels)
print(target_labels)

NameError: name 'plt' is not defined